# Notebook 11: Warm-Start + SRS-Adaptive MAML — MARS

**MAIN CONTRIBUTION (C3):** Combines warm-start initialisation (C1) with SRS-Adaptive inner loop (C2).

**Three differences from Standard MAML (NB07):**
1. **Warm-start init** — loads pretrained GRU backbone from `models/baselines/gru_global_mars.pth` (NB06 output)
2. **Lower outer LR** — `OUTER_LR = 0.0001` (vs 0.001 in NB07) to preserve pretrained knowledge
3. **SRS-Adaptive inner loop** — task-specific `alpha_i = alpha_base * SRS_i` and `K_i = f(SRS_i, tau)`

**Outer loop:** SRS-weighted query loss — `theta <- theta - beta * grad(sum_i SRS_i * L_query(phi_i))`

**Locked hyperparameters (CLAUDE.md):**
```
ALPHA_BASE=0.01  TAU=0.5  K_MIN=3  K_MAX=7
OUTER_LR=0.0001  BATCH_SIZE=32  N_ITER=3000  SEED=20260107
K_SUPPORT=5  Q_QUERY=10
```

**Run order prerequisite:** NB06 (pretrained model) + NB09 (SRS validation) must complete before this notebook.

In [1]:
# [CELL 11-00] Bootstrap: repo root + paths + helpers

import os
import sys
import json
import time
import uuid
import math
import copy
import pickle
import hashlib
import random
from pathlib import Path
from datetime import datetime
from typing import Any, Dict, List, Tuple, Optional
from collections import Counter

import numpy as np
import pandas as pd
import os; os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
import torch
import torch.nn as nn
import torch.nn.functional as F

print(f"[CELL 11-00] start={datetime.now().isoformat(timespec='seconds')}")
print("[CELL 11-00] CWD:", Path.cwd().resolve())


def find_repo_root(start: Path) -> Path:
    """Search upward for PROJECT_STATE.md — single source of truth."""
    start = Path(start).resolve()
    for p in [start, *start.parents]:
        if (p / "PROJECT_STATE.md").exists():
            return p
    raise RuntimeError(
        "Could not find PROJECT_STATE.md in current or parent directories. "
        "Open this notebook from inside the repo."
    )


REPO_ROOT = find_repo_root(Path.cwd())

PATHS = {
    "PROJECT_STATE":  REPO_ROOT / "PROJECT_STATE.md",
    "META_REGISTRY":  REPO_ROOT / "meta.json",
    "DATA_RAW":       REPO_ROOT / "data" / "raw",
    "DATA_INTERIM":   REPO_ROOT / "data" / "interim",
    "DATA_PROCESSED": REPO_ROOT / "data" / "processed",
    "NOTEBOOKS":      REPO_ROOT / "notebooks",
    "REPORTS":        REPO_ROOT / "reports",
    "MODELS":         REPO_ROOT / "models",
    "RUNS":           REPO_ROOT / "runs",
}

for p in PATHS.values():
    Path(p).mkdir(parents=True, exist_ok=True) if str(p).endswith(("reports", "models", "runs")) else None
(PATHS["MODELS"] / "baselines").mkdir(parents=True, exist_ok=True)
(PATHS["MODELS"] / "maml").mkdir(parents=True, exist_ok=True)
(PATHS["REPORTS"]).mkdir(parents=True, exist_ok=True)


def cell_start(cid: str, title: str, **kw) -> float:
    t = time.time()
    print(f"\n[{cid}] {title}")
    print(f"[{cid}] start={datetime.now().isoformat(timespec='seconds')}")
    for k, v in kw.items():
        print(f"[{cid}] {k}={v}")
    return t


def cell_end(cid: str, t0: float, **kw) -> None:
    for k, v in kw.items():
        print(f"[{cid}] {k}={v}")
    print(f"[{cid}] elapsed={time.time()-t0:.2f}s  done")


def write_json_atomic(path: Path, obj: Any, indent: int = 2) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + f".tmp_{uuid.uuid4().hex}")
    tmp.write_text(json.dumps(obj, ensure_ascii=False, indent=indent), encoding="utf-8")
    tmp.replace(path)


def read_json(path: Path) -> Any:
    path = Path(path)
    if not path.exists():
        raise RuntimeError(f"Missing required JSON file: {path}")
    return json.loads(path.read_text(encoding="utf-8"))


print(f"[CELL 11-00] REPO_ROOT={REPO_ROOT}")
print("[CELL 11-00] done")

[CELL 11-00] start=2026-04-10T09:58:00
[CELL 11-00] CWD: /home/user/jamalla/anonymous-users-mooc-session-meta/notebooks/mars
[CELL 11-00] REPO_ROOT=/home/user/jamalla/anonymous-users-mooc-session-meta
[CELL 11-00] done


In [2]:
# [CELL 11-01] Seed + global constants

t0 = cell_start("CELL 11-01", "Seed everything + global constants")

GLOBAL_SEED   = 20260107
NOTEBOOK_NAME = "11_warmstart_srs_adaptive_maml_mars"
DATASET       = "mars"


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    try:
        torch.use_deterministic_algorithms(True)
    except Exception as e:
        print(f"[CELL 11-01] WARN: use_deterministic_algorithms: {e}")


seed_everything(GLOBAL_SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

cell_end("CELL 11-01", t0, seed=GLOBAL_SEED, device=DEVICE)


[CELL 11-01] Seed everything + global constants
[CELL 11-01] start=2026-04-10T09:58:00
[CELL 11-01] seed=20260107
[CELL 11-01] device=cuda
[CELL 11-01] elapsed=0.38s  done


In [3]:
# [CELL 11-02] Run config + data paths
#
# MAIN CONTRIBUTION (C3) — THREE differences from NB07 (Standard MAML):
#   1. OUTER_LR = 0.0001  <- 10x lower than NB07 (0.001) to preserve pretrained init
#   2. warm_start = True  <- loads gru_global_mars.pth
#   3. SRS-Adaptive inner loop (get_task_hyperparams + srs_adaptive_inner_loop)

t0 = cell_start("CELL 11-02", "Run config + paths")

RUN_TAG = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_ID  = uuid.uuid4().hex

OUT_DIR       = PATHS["REPORTS"] / NOTEBOOK_NAME / RUN_TAG
OUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_PATH   = OUT_DIR / "report.json"
CONFIG_PATH   = OUT_DIR / "config.json"
MANIFEST_PATH = OUT_DIR / "manifest.json"

# ── Data paths ────────────────────────────────────────────────
EPISODES_DIR = PATHS["DATA_PROCESSED"] / "mars" / "episodes"
PAIRS_DIR    = PATHS["DATA_PROCESSED"] / "mars" / "pairs"
PAIRS_SRS_DIR = PATHS["DATA_PROCESSED"] / "mars" / "pairs_srs"
VOCAB_DIR    = PATHS["DATA_PROCESSED"] / "mars" / "vocab"
MODELS_DIR   = PATHS["MODELS"] / "maml"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Pretrained backbone from NB06
PRETRAINED_PATH = PATHS["MODELS"] / "baselines" / f"gru_global_{DATASET}.pth"

K_SUPPORT = 5
Q_QUERY   = 10

EPISODES_TRAIN = EPISODES_DIR / f"episodes_train_K{K_SUPPORT}_Q{Q_QUERY}.parquet"
EPISODES_VAL   = EPISODES_DIR / f"episodes_val_K{K_SUPPORT}_Q{Q_QUERY}.parquet"
EPISODES_TEST  = EPISODES_DIR / f"episodes_test_K{K_SUPPORT}_Q{Q_QUERY}.parquet"
PAIRS_TRAIN    = PAIRS_DIR / "pairs_train.parquet"
PAIRS_VAL      = PAIRS_DIR / "pairs_val.parquet"
PAIRS_TEST     = PAIRS_DIR / "pairs_test.parquet"
PAIRS_SRS_PATH = PAIRS_SRS_DIR / "pairs.parquet"
VOCAB_PATH     = VOCAB_DIR / "item2id.json"
CHECKPOINT_PATH = MODELS_DIR / f"maml_warmstart_srs_{DATASET}.pth"
RESULTS_PATH   = OUT_DIR / f"results_{NOTEBOOK_NAME}.json"

# ── Locked hyperparameters (CLAUDE.md — do NOT change) ────────
# SRS-Adaptive parameters
ALPHA_BASE = 0.01    # base inner-loop step size
TAU        = 0.5     # SRS threshold for K selection
K_MIN      = 3       # inner steps for high-SRS sessions (SRS >= tau)
K_MAX      = 7       # inner steps for low-SRS sessions (SRS < tau)
# Outer loop
OUTER_LR   = 0.0001  # <- KEY CHANGE from NB07 (0.001): lower LR preserves pretrained init
BATCH_SIZE = 32
N_ITERATIONS = 3000
VAL_EVERY  = 100
MAX_SEQ_LEN = 50

# ── GRU architecture (locked — same as NB06/NB07/NB08) ───────
GRU_CFG = {
    "embedding_dim": 64,
    "hidden_dim":    128,
    "num_layers":    1,
    "dropout":       0.0,
}

CFG = {
    "notebook":        NOTEBOOK_NAME,
    "dataset":         DATASET,
    "global_seed":     GLOBAL_SEED,
    "device":          DEVICE,
    "K_support":       K_SUPPORT,
    "Q_query":         Q_QUERY,
    "alpha_base":      ALPHA_BASE,
    "tau":             TAU,
    "K_min":           K_MIN,
    "K_max":           K_MAX,
    "outer_lr":        OUTER_LR,
    "batch_size":      BATCH_SIZE,
    "n_iterations":    N_ITERATIONS,
    "val_every":       VAL_EVERY,
    "max_seq_len":     MAX_SEQ_LEN,
    "gru_cfg":         GRU_CFG,
    "pretrained_path": str(PRETRAINED_PATH),
    "warmstart":       True,
    "srs_adaptive":    True,
    "inputs": {
        "episodes_train": str(EPISODES_TRAIN),
        "episodes_val":   str(EPISODES_VAL),
        "episodes_test":  str(EPISODES_TEST),
        "pairs_train":    str(PAIRS_TRAIN),
        "pairs_val":      str(PAIRS_VAL),
        "pairs_test":     str(PAIRS_TEST),
        "pairs_srs":      str(PAIRS_SRS_PATH),
        "vocab":          str(VOCAB_PATH),
    },
}

write_json_atomic(CONFIG_PATH, CFG)

# ── Init report / manifest ────────────────────────────────────
write_json_atomic(REPORT_PATH, {
    "run_id": RUN_ID, "notebook": NOTEBOOK_NAME, "run_tag": RUN_TAG,
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "metrics": {}, "key_findings": [], "sanity_samples": {},
    "data_fingerprints": {}, "notes": [],
})
write_json_atomic(MANIFEST_PATH, {
    "run_id": RUN_ID, "notebook": NOTEBOOK_NAME, "run_tag": RUN_TAG, "artifacts": []
})

# ── Append to meta.json ───────────────────────────────────────
META = PATHS["META_REGISTRY"]
if not META.exists():
    write_json_atomic(META, {"schema_version": 1, "runs": []})
m = read_json(META)
m["runs"].append({"run_id": RUN_ID, "notebook": NOTEBOOK_NAME,
                   "run_tag": RUN_TAG, "out_dir": str(OUT_DIR)})
write_json_atomic(META, m)

print(f"[CELL 11-02] MAIN CONTRIBUTION (C3): Warm-Start + SRS-Adaptive MAML")
print(f"[CELL 11-02] alpha_base={ALPHA_BASE}  tau={TAU}  K_min={K_MIN}  K_max={K_MAX}")
print(f"[CELL 11-02] outer_lr={OUTER_LR}  (NB07 used 0.001 — 10x lower here)")
print(f"[CELL 11-02] batch_size={BATCH_SIZE}  n_iterations={N_ITERATIONS}  val_every={VAL_EVERY}")
print(f"[CELL 11-02] pretrained_path={PRETRAINED_PATH}")
cell_end("CELL 11-02", t0, out_dir=str(OUT_DIR))


[CELL 11-02] Run config + paths
[CELL 11-02] start=2026-04-10T09:58:01
[CELL 11-02] MAIN CONTRIBUTION (C3): Warm-Start + SRS-Adaptive MAML
[CELL 11-02] alpha_base=0.01  tau=0.5  K_min=3  K_max=7
[CELL 11-02] outer_lr=0.0001  (NB07 used 0.001 — 10x lower here)
[CELL 11-02] batch_size=32  n_iterations=3000  val_every=100
[CELL 11-02] pretrained_path=/home/user/jamalla/anonymous-users-mooc-session-meta/models/baselines/gru_global_mars.pth
[CELL 11-02] out_dir=/home/user/jamalla/anonymous-users-mooc-session-meta/reports/11_warmstart_srs_adaptive_maml_mars/20260410_095801
[CELL 11-02] elapsed=0.12s  done


In [4]:
# [CELL 11-03] Load data: vocab, episodes, pairs, pairs_srs

t0 = cell_start("CELL 11-03", "Load vocab + episodes + pairs + pairs_srs")

# ── Guard: required files must exist ─────────────────────────
for label, path in [
    ("vocab",          VOCAB_PATH),
    ("episodes_train", EPISODES_TRAIN),
    ("episodes_val",   EPISODES_VAL),
    ("episodes_test",  EPISODES_TEST),
    ("pairs_train",    PAIRS_TRAIN),
    ("pairs_val",      PAIRS_VAL),
    ("pairs_test",     PAIRS_TEST),
    ("pairs_srs",      PAIRS_SRS_PATH),
]:
    if not Path(path).exists():
        raise RuntimeError(
            f"Missing {label}: {path}\n"
            "Run notebooks 01 -> 02 -> 03 -> 03b -> 04 -> 05 first."
        )

# ── Vocab ────────────────────────────────────────────────────
item2id = read_json(VOCAB_PATH)
id2item  = {int(v): k for k, v in item2id.items()}
n_items    = len(item2id)
print(f"[CELL 11-03] Vocabulary: {n_items} courses")

# ── Episodes ─────────────────────────────────────────────────
episodes_train = pd.read_parquet(EPISODES_TRAIN)
episodes_val   = pd.read_parquet(EPISODES_VAL)
episodes_test  = pd.read_parquet(EPISODES_TEST)

print(f"[CELL 11-03] episodes_train: {len(episodes_train):,} ({episodes_train['user_id'].nunique():,} users)")
print(f"[CELL 11-03] episodes_val:   {len(episodes_val):,} ({episodes_val['user_id'].nunique():,} users)")
print(f"[CELL 11-03] episodes_test:  {len(episodes_test):,} ({episodes_test['user_id'].nunique():,} users)")

# ── Pairs ────────────────────────────────────────────────────
pairs_train = pd.read_parquet(PAIRS_TRAIN)
pairs_val   = pd.read_parquet(PAIRS_VAL)
pairs_test  = pd.read_parquet(PAIRS_TEST)

print(f"[CELL 11-03] pairs_train: {len(pairs_train):,}")
print(f"[CELL 11-03] pairs_val:   {len(pairs_val):,}")
print(f"[CELL 11-03] pairs_test:  {len(pairs_test):,}")

# ── Index pairs by pair_id for fast episode lookup ────────────
pairs_train_idx = pairs_train.set_index("pair_id")
pairs_val_idx   = pairs_val.set_index("pair_id")
pairs_test_idx  = pairs_test.set_index("pair_id")

# ── Pairs with SRS scores ─────────────────────────────────────
pairs_srs = pd.read_parquet(PAIRS_SRS_PATH)
print(f"[CELL 11-03] pairs_srs: {len(pairs_srs):,} rows, columns: {list(pairs_srs.columns)}")

# Build pair_id -> srs_score lookup
# Expected columns: pair_id, srs_score (or similar)
srs_col = None
for candidate in ["srs_score", "srs", "SRS", "reliability"]:
    if candidate in pairs_srs.columns:
        srs_col = candidate
        break
if srs_col is None:
    raise RuntimeError(
        f"Could not find SRS score column in pairs_srs. Available columns: {list(pairs_srs.columns)}\n"
        "Expected one of: srs_score, srs, SRS, reliability. Run NB03b first."
    )

pair_srs_lookup = pairs_srs.set_index("pair_id")[srs_col].to_dict()
print(f"[CELL 11-03] SRS column: '{srs_col}'  |  {len(pair_srs_lookup):,} pair->srs mappings")
print(f"[CELL 11-03] SRS stats: min={min(pair_srs_lookup.values()):.4f}  "
      f"max={max(pair_srs_lookup.values()):.4f}  "
      f"mean={np.mean(list(pair_srs_lookup.values())):.4f}")

cell_end("CELL 11-03", t0, n_items=n_items)


[CELL 11-03] Load vocab + episodes + pairs + pairs_srs
[CELL 11-03] start=2026-04-10T09:58:01
[CELL 11-03] Vocabulary: 745 courses
[CELL 11-03] episodes_train: 111 (9 users)
[CELL 11-03] episodes_val:   6 (6 users)
[CELL 11-03] episodes_test:  17 (17 users)
[CELL 11-03] pairs_train: 693
[CELL 11-03] pairs_val:   545
[CELL 11-03] pairs_test:  1,095
[CELL 11-03] pairs_srs: 2,333 rows, columns: ['pair_id', 'user_id', 'session_id', 'prefix', 'label', 'label_ts_epoch', 'prefix_len', 'srs', 'srs_intensity', 'srs_extent', 'srs_composition']
[CELL 11-03] SRS column: 'srs'  |  2,333 pair->srs mappings
[CELL 11-03] SRS stats: min=0.0791  max=1.0000  mean=0.5817
[CELL 11-03] n_items=745
[CELL 11-03] elapsed=0.46s  done


In [5]:
# [CELL 11-04] Attach SRS to episodes
#
# Each episode has K support pair IDs. We compute the mean SRS score
# across all support pairs as the episode-level SRS.
# This is used to drive the SRS-Adaptive inner loop.

t0 = cell_start("CELL 11-04", "Attach SRS to episodes (mean SRS over support pairs)")

import ast


def get_episode_srs(episode: pd.Series, pair_srs: Dict[Any, float],
                    default_srs: float = 0.5) -> float:
    """
    Compute mean SRS score for an episode from its support pair IDs.

    Args:
        episode:     row from episodes DataFrame
        pair_srs:    dict mapping pair_id -> srs_score
        default_srs: fallback if no SRS found (neutral = 0.5)

    Returns:
        float in [0, 1] — mean SRS across support pairs
    """
    sup_ids = episode["support_pair_ids"]
    if isinstance(sup_ids, str):
        sup_ids = ast.literal_eval(sup_ids)
    scores = [pair_srs.get(pid, default_srs) for pid in sup_ids]
    return float(np.mean(scores)) if scores else default_srs


# Attach SRS to all episode splits
episodes_train["episode_srs"] = episodes_train.apply(
    lambda row: get_episode_srs(row, pair_srs_lookup), axis=1
)
episodes_val["episode_srs"] = episodes_val.apply(
    lambda row: get_episode_srs(row, pair_srs_lookup), axis=1
)
episodes_test["episode_srs"] = episodes_test.apply(
    lambda row: get_episode_srs(row, pair_srs_lookup), axis=1
)

print(f"[CELL 11-04] Train SRS: mean={episodes_train['episode_srs'].mean():.4f}  "
      f"std={episodes_train['episode_srs'].std():.4f}  "
      f"min={episodes_train['episode_srs'].min():.4f}  "
      f"max={episodes_train['episode_srs'].max():.4f}")
print(f"[CELL 11-04] Val   SRS: mean={episodes_val['episode_srs'].mean():.4f}  "
      f"std={episodes_val['episode_srs'].std():.4f}")
print(f"[CELL 11-04] Test  SRS: mean={episodes_test['episode_srs'].mean():.4f}  "
      f"std={episodes_test['episode_srs'].std():.4f}")

# Tier breakdown for training set
low    = (episodes_train["episode_srs"] < 0.33).mean()
medium = ((episodes_train["episode_srs"] >= 0.33) & (episodes_train["episode_srs"] < 0.66)).mean()
high   = (episodes_train["episode_srs"] >= 0.66).mean()
print(f"[CELL 11-04] Train SRS tiers:  low={low*100:.1f}%  medium={medium*100:.1f}%  high={high*100:.1f}%")
print(f"[CELL 11-04] Episodes with SRS < TAU ({TAU}): {(episodes_train['episode_srs'] < TAU).mean()*100:.1f}% "
      f"-> will use K_MAX={K_MAX} inner steps")
print(f"[CELL 11-04] Episodes with SRS >= TAU ({TAU}): {(episodes_train['episode_srs'] >= TAU).mean()*100:.1f}% "
      f"-> will use K_MIN={K_MIN} inner steps")

cell_end("CELL 11-04", t0)


[CELL 11-04] Attach SRS to episodes (mean SRS over support pairs)
[CELL 11-04] start=2026-04-10T09:58:01
[CELL 11-04] Train SRS: mean=0.8612  std=0.1706  min=0.3397  max=1.0000
[CELL 11-04] Val   SRS: mean=0.8033  std=0.2632
[CELL 11-04] Test  SRS: mean=0.8139  std=0.2339
[CELL 11-04] Train SRS tiers:  low=0.0%  medium=11.7%  high=88.3%
[CELL 11-04] Episodes with SRS < TAU (0.5): 6.3% -> will use K_MAX=7 inner steps
[CELL 11-04] Episodes with SRS >= TAU (0.5): 93.7% -> will use K_MIN=3 inner steps
[CELL 11-04] elapsed=0.01s  done


In [6]:
# [CELL 11-05] GRURecommender model definition + warm-start initialisation
#
# Architecture is IDENTICAL to NB06/NB07/NB08.
# After defining the model, this cell loads the pretrained weights from NB06.
# This is KEY DIFFERENCE #1 and #2 from Standard MAML (NB07).

t0 = cell_start("CELL 11-05", "Define GRURecommender + load pretrained weights (warm-start)")


class GRURecommender(nn.Module):
    """GRU-based sequential recommender.

    Architecture (matches NB06/NB07/NB08):
        Embedding(n_items, embedding_dim) -> GRU(embedding_dim, hidden_dim) -> Linear(hidden_dim, n_items)
    """

    def __init__(
        self,
        n_items: int,
        embedding_dim: int = 64,
        hidden_dim: int = 128,
        num_layers: int = 1,
        dropout: float = 0.0,
    ):
        super().__init__()
        self.n_items       = n_items
        self.embedding_dim = embedding_dim
        self.hidden_dim    = hidden_dim
        self.num_layers    = num_layers

        self.embedding = nn.Embedding(n_items + 1, embedding_dim, padding_idx=0)
        gru_dropout = dropout if num_layers > 1 else 0.0
        self.gru = nn.GRU(
            embedding_dim, hidden_dim, num_layers,
            batch_first=True, dropout=gru_dropout
        )
        self.fc = nn.Linear(hidden_dim, n_items)

    def forward(self, seq: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
        """
        Args:
            seq:     (batch, max_len) padded item-id sequences
            lengths: (batch,) actual sequence lengths
        Returns:
            logits: (batch, n_items)
        """
        emb = self.embedding(seq)                        # (B, L, E)
        packed = nn.utils.rnn.pack_padded_sequence(
            emb, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        _, h_n = self.gru(packed)                        # h_n: (layers, B, H)
        h_last = h_n[-1]                                 # (B, H)
        return self.fc(h_last)                           # (B, n_items)


# Quick sanity check
_m = GRURecommender(n_items=100, **GRU_CFG)
_x = torch.randint(1, 100, (4, 10))
_l = torch.tensor([10, 8, 5, 3])
_o = _m(_x, _l)
assert _o.shape == (4, 100), f"Expected (4,100) got {_o.shape}"
del _m, _x, _l, _o
print("[CELL 11-05] GRURecommender sanity check passed.")

# ── Build meta-model ──────────────────────────────────────────
seed_everything(GLOBAL_SEED)   # deterministic init

meta_model = GRURecommender(
    n_items=n_items,
    embedding_dim=GRU_CFG["embedding_dim"],
    hidden_dim=GRU_CFG["hidden_dim"],
    num_layers=GRU_CFG["num_layers"],
    dropout=GRU_CFG["dropout"],
).to(DEVICE)

n_params = sum(p.numel() for p in meta_model.parameters())
print(f"[CELL 11-05] Meta-model parameters: {n_params:,}")

# ── Warm-start: load pretrained GRU backbone (KEY DIFFERENCE #1) ─
pretrained_path = PRETRAINED_PATH
if not pretrained_path.exists():
    raise RuntimeError(
        f"Pretrained model not found: {pretrained_path}. Run NB06 first."
    )
meta_model.load_state_dict(torch.load(pretrained_path, map_location=DEVICE))
print(f"[CELL 11-05] Loaded pretrained weights from {pretrained_path}")
print(f"[CELL 11-05] Warm-start init complete.")
print(f"[CELL 11-05] NOTE: outer_lr={OUTER_LR} (KEY DIFFERENCE #2 — 10x lower than NB07) to preserve pretrained knowledge.")

cell_end("CELL 11-05", t0, n_params=n_params)


[CELL 11-05] Define GRURecommender + load pretrained weights (warm-start)
[CELL 11-05] start=2026-04-10T09:58:01
[CELL 11-05] GRURecommender sanity check passed.
[CELL 11-05] Meta-model parameters: 218,345
[CELL 11-05] Loaded pretrained weights from /home/user/jamalla/anonymous-users-mooc-session-meta/models/baselines/gru_global_mars.pth
[CELL 11-05] Warm-start init complete.
[CELL 11-05] NOTE: outer_lr=0.0001 (KEY DIFFERENCE #2 — 10x lower than NB07) to preserve pretrained knowledge.
[CELL 11-05] n_params=218345
[CELL 11-05] elapsed=1.83s  done


In [7]:
# [CELL 11-06] functional_forward — uses torch.func.functional_call (fast CUDA GRU)
t0 = cell_start("CELL 11-06", "Define functional_forward for FOMAML")

from torch.func import functional_call

def functional_forward(model, params, seqs, lengths):
    """
    Functional forward using torch.func.functional_call.
    Runs the model native optimized CUDA GRU with the given params dict.
    Vastly faster than manual step-by-step GRU loop.
    """
    return functional_call(model, params, (seqs, lengths))


cell_end("CELL 11-06", t0)



[CELL 11-06] Define functional_forward for FOMAML
[CELL 11-06] start=2026-04-10T09:58:03
[CELL 11-06] elapsed=0.00s  done


In [8]:
# [CELL 11-07] SRS-Adaptive functions (KEY DIFFERENCE #3 from Standard MAML)
#
# IDENTICAL to NB10 (SRS-Adaptive MAML without warm-start).
# get_task_hyperparams, srs_adaptive_inner_loop, outer_step are UNCHANGED.
#
# Formula:
#   alpha_i = alpha_base * SRS_i
#   K_i     = K_min if SRS_i >= tau else K_max
#   theta  <- theta - beta * grad(sum_i SRS_i * L_query(phi_i))

t0 = cell_start("CELL 11-07", "Define SRS-Adaptive functions (get_task_hyperparams, inner loop, outer step)")


def get_task_hyperparams(
    srs_i: float,
    alpha_base: float = ALPHA_BASE,
    tau: float = TAU,
    k_min: int = K_MIN,
    k_max: int = K_MAX,
) -> Tuple[float, int]:
    """
    Compute task-specific inner-loop hyperparameters from session SRS score.

    alpha_i = alpha_base * SRS_i
    K_i     = K_min if SRS_i >= tau else K_max

    High SRS -> larger alpha, fewer steps (reliable session, fast adaptation)
    Low  SRS -> smaller alpha, more steps  (noisy session, careful adaptation)

    Args:
        srs_i:      SRS score in [0, 1] for this task
        alpha_base: base inner-loop step size
        tau:        SRS threshold for K selection
        k_min:      inner steps for high-SRS sessions (SRS >= tau)
        k_max:      inner steps for low-SRS sessions  (SRS <  tau)

    Returns:
        (alpha_i, K_i)
    """
    alpha_i = alpha_base * srs_i
    K_i     = k_min if srs_i >= tau else k_max
    return float(alpha_i), int(K_i)


def srs_adaptive_inner_loop(
    model: GRURecommender,
    params: Dict[str, torch.Tensor],
    support_seqs: torch.Tensor,
    support_lengths: torch.Tensor,
    support_labels: torch.Tensor,
    alpha_i: float,
    K_i: int,
) -> Dict[str, torch.Tensor]:
    """
    Standard MAML inner loop with task-specific alpha and K.
    alpha_i and K_i are computed from the session SRS score.
    Updates params in-place (functional) and returns adapted phi_i.

    Args:
        model:           GRURecommender (architecture metadata only)
        params:          current parameter dict (theta or partial phi)
        support_seqs:    (K, max_len) support sequences
        support_lengths: (K,) support sequence lengths
        support_labels:  (K,) support labels
        alpha_i:         task-specific inner LR = alpha_base * SRS_i
        K_i:             task-specific inner steps

    Returns:
        phi_i: adapted parameter dict
    """
    for step in range(K_i):
        logits = functional_forward(model, params, support_seqs, support_lengths)
        loss   = F.cross_entropy(logits, support_labels)
        grads  = torch.autograd.grad(loss, params.values(), create_graph=False)
        params = {n: p - alpha_i * g
                  for (n, p), g in zip(params.items(), grads)}
    return params


def outer_step(
    meta_model: GRURecommender,
    batch_tasks: List[Dict],
    meta_optimizer: torch.optim.Optimizer,
) -> float:
    """
    One outer-loop step with SRS-weighted gradient.

    theta <- theta - beta * grad_theta( sum_i SRS_i * L_query(phi_i) )

    High-reliability tasks (SRS_i close to 1) have stronger influence on theta.
    Low-reliability  tasks (SRS_i close to 0) contribute less to the outer loss.

    Args:
        meta_model:     current meta-model (theta)
        batch_tasks:    list of task dicts, each with keys:
                            srs, support_seqs, support_lengths, support_labels,
                            query_seqs, query_lengths, query_labels
        meta_optimizer: outer-loop Adam optimizer

    Returns:
        total outer loss (float) for logging
    """
    meta_optimizer.zero_grad()
    total_loss = torch.tensor(0.0, requires_grad=True)

    for task in batch_tasks:
        srs_i        = task["srs"]                       # float in [0, 1]
        alpha_i, K_i = get_task_hyperparams(srs_i)
        params       = {n: p.clone() for n, p in meta_model.named_parameters()}
        phi_i        = srs_adaptive_inner_loop(
                            meta_model, params,
                            task["support_seqs"],
                            task["support_lengths"],
                            task["support_labels"],
                            alpha_i, K_i)
        query_logits = functional_forward(meta_model, phi_i,
                            task["query_seqs"],
                            task["query_lengths"])
        query_loss   = F.cross_entropy(query_logits, task["query_labels"])
        total_loss   = total_loss + srs_i * query_loss   # SRS-weighted

    total_loss.backward()
    meta_optimizer.step()
    return total_loss.item()


# Quick unit test for get_task_hyperparams
_a, _k = get_task_hyperparams(0.8)
assert _a == ALPHA_BASE * 0.8, f"alpha_i mismatch: {_a}"
assert _k == K_MIN, f"K_i should be K_MIN for high SRS: {_k}"
_a2, _k2 = get_task_hyperparams(0.3)
assert _k2 == K_MAX, f"K_i should be K_MAX for low SRS: {_k2}"
print(f"[CELL 11-07] get_task_hyperparams: high-SRS alpha={_a:.4f} K={_k} | low-SRS alpha={_a2:.4f} K={_k2}")
del _a, _k, _a2, _k2

cell_end("CELL 11-07", t0)


[CELL 11-07] Define SRS-Adaptive functions (get_task_hyperparams, inner loop, outer step)
[CELL 11-07] start=2026-04-10T09:58:03
[CELL 11-07] get_task_hyperparams: high-SRS alpha=0.0080 K=3 | low-SRS alpha=0.0030 K=7
[CELL 11-07] elapsed=0.00s  done


In [9]:
# [CELL 11-08] Evaluation metrics: HR@1/5/10, NDCG@10, MRR
#
# IDENTICAL to NB07/NB08. Primary metrics: HR@10 and NDCG@10.

t0 = cell_start("CELL 11-08", "Define evaluation metrics")


def ndcg_at_k(ranked_items: np.ndarray, true_item: int, k: int = 10) -> float:
    """NDCG@K = 1/log2(rank+2) if true_item in top-K, else 0."""
    for i, item in enumerate(ranked_items[:k]):
        if item == true_item:
            return 1.0 / math.log2(i + 2)
    return 0.0


def compute_metrics(
    scores: np.ndarray,
    labels: np.ndarray,
) -> Dict[str, float]:
    """
    Compute HR@1, HR@5, HR@10, NDCG@10, MRR.

    Args:
        scores: (n_samples, n_items) float score matrix (higher = more relevant)
        labels: (n_samples,) int true item indices

    Returns:
        dict with keys: hr1, hr5, hr10, ndcg10, mrr, n
    """
    n = len(labels)
    ranked = np.argsort(-scores, axis=1)   # descending rank

    hr1_vals    = []
    hr5_vals    = []
    hr10_vals   = []
    ndcg10_vals = []
    mrr_vals    = []

    for i in range(n):
        true     = int(labels[i])
        rank_arr = ranked[i]

        hr1_vals.append(1.0 if rank_arr[0] == true else 0.0)
        hr5_vals.append(1.0 if true in rank_arr[:5] else 0.0)
        hr10_vals.append(1.0 if true in rank_arr[:10] else 0.0)
        ndcg10_vals.append(ndcg_at_k(rank_arr, true, k=10))

        pos = np.where(rank_arr == true)[0]
        mrr_vals.append(1.0 / (pos[0] + 1) if len(pos) > 0 else 0.0)

    return {
        "hr1":    float(np.mean(hr1_vals)),
        "hr5":    float(np.mean(hr5_vals)),
        "hr10":   float(np.mean(hr10_vals)),
        "ndcg10": float(np.mean(ndcg10_vals)),
        "mrr":    float(np.mean(mrr_vals)),
        "n":      n,
    }


cell_end("CELL 11-08", t0)


[CELL 11-08] Define evaluation metrics
[CELL 11-08] start=2026-04-10T09:58:04
[CELL 11-08] elapsed=0.00s  done


In [10]:
# [CELL 11-09] Episode collation helpers: pad_sequences + get_episode_data_with_srs
#
# get_episode_data_with_srs extends the NB07/NB08 helper by also returning
# the episode-level SRS score, which drives the SRS-Adaptive inner loop.

t0 = cell_start("CELL 11-09", "Define pad_sequences + get_episode_data_with_srs")


def pad_sequences(
    sequences: List[List[int]],
    max_len: int,
    pad_value: int = 0,
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Pad a list of variable-length sequences to max_len.

    Returns:
        padded:  (batch, max_len) LongTensor
        lengths: (batch,)         LongTensor of actual lengths (clipped to max_len)
    """
    batch  = len(sequences)
    padded = torch.full((batch, max_len), pad_value, dtype=torch.long)
    lengths = torch.zeros(batch, dtype=torch.long)
    for i, seq in enumerate(sequences):
        seq = seq[:max_len]
        l   = len(seq)
        if l > 0:
            padded[i, :l] = torch.tensor(seq, dtype=torch.long)
        lengths[i] = max(l, 1)   # avoid zero-length (GRU pack requirement)
    return padded, lengths


def get_episode_data(
    episode: pd.Series,
    pairs_idx: pd.DataFrame,
    max_seq_len: int = MAX_SEQ_LEN,
    device: str = "cpu",
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor,
           torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    Fetch and tensorise support/query batches for one episode.

    Returns:
        sup_seqs, sup_lengths, sup_labels  -- support batch
        qry_seqs, qry_lengths, qry_labels  -- query batch
    """
    sup_ids = episode["support_pair_ids"]
    qry_ids = episode["query_pair_ids"]

    if isinstance(sup_ids, str):
        import ast
        sup_ids = ast.literal_eval(sup_ids)
        qry_ids = ast.literal_eval(qry_ids)

    def fetch_batch(pair_ids):
        rows  = pairs_idx.loc[pair_ids]
        seqs  = []
        for prefix in rows["prefix"]:
            if isinstance(prefix, str):
                import ast
                prefix = ast.literal_eval(prefix)
            seqs.append(list(prefix))
        labels = rows["label"].tolist()
        return seqs, labels

    sup_seqs_list, sup_labels_list = fetch_batch(sup_ids)
    qry_seqs_list, qry_labels_list = fetch_batch(qry_ids)

    sup_seqs, sup_lengths = pad_sequences(sup_seqs_list, max_seq_len)
    qry_seqs, qry_lengths = pad_sequences(qry_seqs_list, max_seq_len)

    sup_labels = torch.tensor(sup_labels_list, dtype=torch.long)
    qry_labels = torch.tensor(qry_labels_list, dtype=torch.long)

    return (
        sup_seqs.to(device),  sup_lengths.to(device),  sup_labels.to(device),
        qry_seqs.to(device),  qry_lengths.to(device),  qry_labels.to(device),
    )


cell_end("CELL 11-09", t0)


[CELL 11-09] Define pad_sequences + get_episode_data_with_srs
[CELL 11-09] start=2026-04-10T09:58:04
[CELL 11-09] elapsed=0.00s  done


In [11]:
# [CELL 11-10] Training loop — 3000 iterations, val every 100, save best HR@10 checkpoint
#
# This is the MAIN CONTRIBUTION training loop:
#   - Warm-start init already applied in CELL 11-05
#   - OUTER_LR = 0.0001 (KEY DIFFERENCE #2: preserves pretrained knowledge)
#   - SRS-Adaptive inner loop via get_task_hyperparams + srs_adaptive_inner_loop (KEY DIFFERENCE #3)
#   - SRS-weighted outer loss: sum_i SRS_i * L_query(phi_i)

t0_train = cell_start(
    "CELL 11-10", "Warm-Start + SRS-Adaptive MAML training",
    n_iterations=N_ITERATIONS, batch_size=BATCH_SIZE,
    alpha_base=ALPHA_BASE, tau=TAU, K_min=K_MIN, K_max=K_MAX,
    outer_lr=OUTER_LR, warmstart=True, srs_adaptive=True
)

meta_optimizer = torch.optim.Adam(meta_model.parameters(), lr=OUTER_LR)

# ── Prepare training index ────────────────────────────────────
train_indices = list(range(len(episodes_train)))

# ── Tracking ─────────────────────────────────────────────────
train_losses  = []
val_hr10_hist = []
best_val_hr10 = -1.0
best_iter     = 0
best_state    = None

# ── Outer loop ───────────────────────────────────────────────
for iteration in range(1, N_ITERATIONS + 1):
    meta_model.train()
    meta_optimizer.zero_grad()

    # Sample batch of task indices
    batch_idx = np.random.choice(train_indices, size=min(BATCH_SIZE, len(train_indices)), replace=False)
    batch_episodes = episodes_train.iloc[batch_idx]

    outer_loss = torch.tensor(0.0, device=DEVICE)
    n_valid    = 0

    for _, episode in batch_episodes.iterrows():
        try:
            sup_seqs, sup_lengths, sup_labels, \
            qry_seqs, qry_lengths, qry_labels = get_episode_data(
                episode, pairs_train_idx, MAX_SEQ_LEN, DEVICE
            )
        except (KeyError, ValueError):
            continue

        # ── SRS-Adaptive inner loop (KEY DIFFERENCE #3) ───────
        srs_i        = float(episode["episode_srs"])
        alpha_i, K_i = get_task_hyperparams(srs_i)
        params       = {n: p.clone() for n, p in meta_model.named_parameters()}
        phi_i        = srs_adaptive_inner_loop(
                            meta_model, params,
                            sup_seqs, sup_lengths, sup_labels,
                            alpha_i, K_i)

        # ── SRS-weighted query loss ───────────────────────────
        # theta <- theta - beta * grad( sum_i SRS_i * L_query(phi_i) )
        qry_logits = functional_forward(meta_model, phi_i, qry_seqs, qry_lengths)
        loss_q     = F.cross_entropy(qry_logits, qry_labels)
        outer_loss = outer_loss + srs_i * loss_q   # SRS-weighted
        n_valid   += 1

    if n_valid == 0:
        continue

    (outer_loss / n_valid).backward()
    torch.nn.utils.clip_grad_norm_(meta_model.parameters(), max_norm=5.0)
    meta_optimizer.step()

    train_losses.append(float(outer_loss.item() / n_valid))

    # ── Validation every VAL_EVERY iterations ────────────────
    if iteration % VAL_EVERY == 0:
        meta_model.train()  # keep train mode — inner loop needs cuDNN backward
        val_scores_all = []
        val_labels_all = []

        for _, ep in episodes_val.iterrows():
            try:
                sup_s, sup_l, sup_lb, qry_s, qry_l, qry_lb = get_episode_data(
                    ep, pairs_val_idx, MAX_SEQ_LEN, DEVICE
                )
            except (KeyError, ValueError):
                continue

            # Adapt on support with SRS-Adaptive inner loop
            srs_ep       = float(ep["episode_srs"])
            alpha_v, K_v = get_task_hyperparams(srs_ep)
            params_v     = {n: p.clone() for n, p in meta_model.named_parameters()}
            for _step in range(K_v):
                lg_v   = functional_forward(meta_model, params_v, sup_s, sup_l)
                ls_v   = F.cross_entropy(lg_v, sup_lb)
                gv     = torch.autograd.grad(ls_v, list(params_v.values()), create_graph=False)
                params_v = {n: p - alpha_v * g for (n, p), g in zip(params_v.items(), gv)}

            with torch.no_grad():

                qry_lg = functional_forward(meta_model, params_v, qry_s, qry_l)
            val_scores_all.append(qry_lg.detach().cpu().numpy())
            val_labels_all.extend(qry_lb.cpu().tolist())

        if len(val_labels_all) > 0:
            val_scores_np = np.vstack(val_scores_all)
            val_labels_np = np.array(val_labels_all)
            val_m = compute_metrics(val_scores_np, val_labels_np)
            val_hr10_hist.append((iteration, val_m["hr10"]))

            if val_m["hr10"] > best_val_hr10:
                best_val_hr10 = val_m["hr10"]
                best_iter     = iteration
                best_state    = copy.deepcopy(meta_model.state_dict())
                torch.save(best_state, CHECKPOINT_PATH)

            print(
                f"[CELL 11-10] iter={iteration:4d} "
                f"train_loss={train_losses[-1]:.4f}  "
                f"val_HR@10={val_m['hr10']*100:.2f}%  "
                f"best_HR@10={best_val_hr10*100:.2f}%@{best_iter}"
            )

print(f"\n[CELL 11-10] Training complete.")
print(f"[CELL 11-10] Best val HR@10: {best_val_hr10*100:.2f}% at iteration {best_iter}")
print(f"[CELL 11-10] Checkpoint saved: {CHECKPOINT_PATH}")

cell_end("CELL 11-10", t0_train, best_iter=best_iter, best_val_hr10=f"{best_val_hr10*100:.2f}%")



[CELL 11-10] Warm-Start + SRS-Adaptive MAML training
[CELL 11-10] start=2026-04-10T09:58:04
[CELL 11-10] n_iterations=3000
[CELL 11-10] batch_size=32
[CELL 11-10] alpha_base=0.01
[CELL 11-10] tau=0.5
[CELL 11-10] K_min=3
[CELL 11-10] K_max=7
[CELL 11-10] outer_lr=0.0001
[CELL 11-10] warmstart=True
[CELL 11-10] srs_adaptive=True
[CELL 11-10] iter= 100 train_loss=2.7800  val_HR@10=3.33%  best_HR@10=3.33%@100
[CELL 11-10] iter= 200 train_loss=2.2068  val_HR@10=3.33%  best_HR@10=3.33%@100
[CELL 11-10] iter= 300 train_loss=1.3232  val_HR@10=3.33%  best_HR@10=3.33%@100
[CELL 11-10] iter= 400 train_loss=0.9339  val_HR@10=1.67%  best_HR@10=3.33%@100
[CELL 11-10] iter= 500 train_loss=0.6961  val_HR@10=1.67%  best_HR@10=3.33%@100
[CELL 11-10] iter= 600 train_loss=0.4123  val_HR@10=1.67%  best_HR@10=3.33%@100
[CELL 11-10] iter= 700 train_loss=0.3402  val_HR@10=1.67%  best_HR@10=3.33%@100
[CELL 11-10] iter= 800 train_loss=0.2364  val_HR@10=1.67%  best_HR@10=3.33%@100
[CELL 11-10] iter= 900 train_

In [12]:
# [CELL 11-11] Test evaluation — restore best checkpoint, evaluate on test episodes

t0 = cell_start("CELL 11-11", "Test evaluation (best checkpoint)")

# Load best checkpoint
if not CHECKPOINT_PATH.exists():
    raise RuntimeError(f"Checkpoint missing: {CHECKPOINT_PATH}. Run training cell first.")

meta_model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=DEVICE))
meta_model.train()  # keep train mode — inner loop needs cuDNN backward
print(f"[CELL 11-11] Loaded best checkpoint from iter {best_iter}: {CHECKPOINT_PATH}")

test_scores_all = []
test_labels_all = []
n_skipped = 0

for _, ep in episodes_test.iterrows():
    try:
        sup_s, sup_l, sup_lb, qry_s, qry_l, qry_lb = get_episode_data(
            ep, pairs_test_idx, MAX_SEQ_LEN, DEVICE
        )
    except (KeyError, ValueError):
        n_skipped += 1
        continue

    # Adapt on support set with SRS-Adaptive inner loop (K=5 support)
    srs_ep       = float(ep["episode_srs"])
    alpha_t, K_t = get_task_hyperparams(srs_ep)
    params_t     = {n: p.clone() for n, p in meta_model.named_parameters()}
    for _step in range(K_t):
        with torch.enable_grad():
            lg_t = functional_forward(meta_model, params_t, sup_s, sup_l)
            ls_t = F.cross_entropy(lg_t, sup_lb)
            gt   = torch.autograd.grad(ls_t, list(params_t.values()), create_graph=False)
        params_t = {n: p - alpha_t * g for (n, p), g in zip(params_t.items(), gt)}

    with torch.no_grad():
        qry_lg = functional_forward(meta_model, params_t, qry_s, qry_l)
    test_scores_all.append(qry_lg.detach().cpu().numpy())
    test_labels_all.extend(qry_lb.cpu().tolist())

print(f"[CELL 11-11] episodes evaluated: {len(test_scores_all)}, skipped: {n_skipped}")

test_scores_np = np.vstack(test_scores_all)
test_labels_np = np.array(test_labels_all)

test_metrics = compute_metrics(test_scores_np, test_labels_np)

test_hr1    = test_metrics["hr1"]
test_hr5    = test_metrics["hr5"]
test_hr10   = test_metrics["hr10"]
test_ndcg10 = test_metrics["ndcg10"]
test_mrr    = test_metrics["mrr"]

print(f"\n[CELL 11-11] TEST RESULTS (K={K_SUPPORT} support, Q={Q_QUERY} query):")
print(f"  HR@1    = {test_hr1*100:.2f}%")
print(f"  HR@5    = {test_hr5*100:.2f}%")
print(f"  HR@10   = {test_hr10*100:.2f}%   <- PRIMARY")
print(f"  NDCG@10 = {test_ndcg10*100:.2f}%   <- PRIMARY")
print(f"  MRR     = {test_mrr*100:.2f}%")

cell_end("CELL 11-11", t0, n_test_samples=len(test_labels_all))


[CELL 11-11] Test evaluation (best checkpoint)
[CELL 11-11] start=2026-04-10T10:21:51
[CELL 11-11] Loaded best checkpoint from iter 100: /home/user/jamalla/anonymous-users-mooc-session-meta/models/maml/maml_warmstart_srs_mars.pth
[CELL 11-11] episodes evaluated: 17, skipped: 0

[CELL 11-11] TEST RESULTS (K=5 support, Q=10 query):
  HR@1    = 17.65%
  HR@5    = 25.29%
  HR@10   = 27.06%   <- PRIMARY
  NDCG@10 = 21.83%   <- PRIMARY
  MRR     = 20.72%
[CELL 11-11] n_test_samples=170
[CELL 11-11] elapsed=0.29s  done


In [13]:
# [CELL 11-12] Standard result block (CLAUDE.md mandatory format)

print("=" * 55)
print(f"RESULTS — {NOTEBOOK_NAME} — {DATASET}")
print("=" * 55)
print(f"Protocol:      K={K_SUPPORT} support, Q={Q_QUERY} query")
print(f"Test episodes: {len(episodes_test)}")
print(f"Seed:          {GLOBAL_SEED}")
print()
print(f"HR@1:          {test_hr1*100:.2f}%")
print(f"HR@5:          {test_hr5*100:.2f}%")
print(f"HR@10:         {test_hr10*100:.2f}%   <- PRIMARY")
print(f"NDCG@10:       {test_ndcg10*100:.2f}%   <- PRIMARY")
print(f"MRR:           {test_mrr*100:.2f}%")
print()
print(f"Best val iter: {best_iter}")
print(f"Best val HR@10:{best_val_hr10*100:.2f}%")
print("=" * 55)
print()
print("CONFIG DIFFERENCES FROM NB07 (Standard MAML):")
print(f"  outer_lr     = {OUTER_LR}  (NB07: 0.001 — 10x lower here)")
print(f"  warm_start   = True  (loaded gru_global_{DATASET}.pth from NB06)")
print(f"  srs_adaptive = True  (alpha_i=alpha_base*SRS_i, K_i=K_min/K_max per TAU)")
print(f"  tau={TAU}  K_min={K_MIN}  K_max={K_MAX}  alpha_base={ALPHA_BASE}")

RESULTS — 11_warmstart_srs_adaptive_maml_mars — mars
Protocol:      K=5 support, Q=10 query
Test episodes: 17
Seed:          20260107

HR@1:          17.65%
HR@5:          25.29%
HR@10:         27.06%   <- PRIMARY
NDCG@10:       21.83%   <- PRIMARY
MRR:           20.72%

Best val iter: 100
Best val HR@10:3.33%

CONFIG DIFFERENCES FROM NB07 (Standard MAML):
  outer_lr     = 0.0001  (NB07: 0.001 — 10x lower here)
  warm_start   = True  (loaded gru_global_mars.pth from NB06)
  srs_adaptive = True  (alpha_i=alpha_base*SRS_i, K_i=K_min/K_max per TAU)
  tau=0.5  K_min=3  K_max=7  alpha_base=0.01


In [14]:
# [CELL 11-13] Save results JSON + update report and manifest

t0 = cell_start("CELL 11-13", "Save results JSON")

results = {
    "run_id":     RUN_ID,
    "notebook":   NOTEBOOK_NAME,
    "dataset":    DATASET,
    "run_tag":    RUN_TAG,
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "config": {
        "K_support":       K_SUPPORT,
        "Q_query":         Q_QUERY,
        "alpha_base":      ALPHA_BASE,
        "tau":             TAU,
        "K_min":           K_MIN,
        "K_max":           K_MAX,
        "outer_lr":        OUTER_LR,
        "batch_size":      BATCH_SIZE,
        "n_iterations":    N_ITERATIONS,
        "global_seed":     GLOBAL_SEED,
        "gru_cfg":         GRU_CFG,
        "warmstart":       True,
        "srs_adaptive":    True,
        "pretrained_path": str(PRETRAINED_PATH),
    },
    "test_metrics": {
        "hr1":    test_hr1,
        "hr5":    test_hr5,
        "hr10":   test_hr10,
        "ndcg10": test_ndcg10,
        "mrr":    test_mrr,
        "n":      int(len(test_labels_all)),
    },
    "val_best": {
        "best_iter":     best_iter,
        "best_val_hr10": best_val_hr10,
    },
    "checkpoint": str(CHECKPOINT_PATH),
}

write_json_atomic(RESULTS_PATH, results)
print(f"[CELL 11-13] Results saved: {RESULTS_PATH}")

# Update report.json
report = read_json(REPORT_PATH)
report["metrics"] = results["test_metrics"]
report["key_findings"] = [
    f"MAIN CONTRIBUTION (C3) — Warm-Start + SRS-Adaptive MAML: HR@10={test_hr10*100:.2f}%, NDCG@10={test_ndcg10*100:.2f}%",
    f"Best val HR@10={best_val_hr10*100:.2f}% at iteration {best_iter}",
    f"Hyperparams: alpha_base={ALPHA_BASE}, tau={TAU}, K_min={K_MIN}, K_max={K_MAX}, outer_lr={OUTER_LR}, batch={BATCH_SIZE}, iters={N_ITERATIONS}",
    f"Warm-start from: {PRETRAINED_PATH}",
    f"SRS-Adaptive inner loop: alpha_i=alpha_base*SRS_i, K_i=K_min if SRS>=tau else K_max",
    f"Outer loop: SRS-weighted loss sum_i SRS_i * L_query(phi_i)",
    f"Test episodes evaluated: {len(test_labels_all)}, skipped: {n_skipped}",
]
write_json_atomic(REPORT_PATH, report)

# Update manifest.json
manifest = read_json(MANIFEST_PATH)
manifest["artifacts"].extend([
    {"type": "results_json",  "path": str(RESULTS_PATH)},
    {"type": "checkpoint",    "path": str(CHECKPOINT_PATH)},
    {"type": "report",        "path": str(REPORT_PATH)},
    {"type": "config",        "path": str(CONFIG_PATH)},
])
write_json_atomic(MANIFEST_PATH, manifest)

cell_end("CELL 11-13", t0)


[CELL 11-13] Save results JSON
[CELL 11-13] start=2026-04-10T10:21:51
[CELL 11-13] Results saved: /home/user/jamalla/anonymous-users-mooc-session-meta/reports/11_warmstart_srs_adaptive_maml_mars/20260410_095801/results_11_warmstart_srs_adaptive_maml_mars.json
[CELL 11-13] elapsed=0.05s  done


In [15]:
# [CELL 11-14] Update PROJECT_STATE.md with results

t0 = cell_start("CELL 11-14", "Update PROJECT_STATE.md")

state_path = PATHS["PROJECT_STATE"]

result_block = (
    f"\n\n## {NOTEBOOK_NAME} — Results\n"
    f"Run: {RUN_TAG}  |  Seed: {GLOBAL_SEED}\n"
    f"Protocol: K={K_SUPPORT} support, Q={Q_QUERY} query | Test episodes: {len(episodes_test)}\n"
    f"MAIN CONTRIBUTION (C3): Warm-Start + SRS-Adaptive MAML | outer_lr={OUTER_LR}\n"
    f"alpha_base={ALPHA_BASE}  tau={TAU}  K_min={K_MIN}  K_max={K_MAX}\n"
    f"\n"
    f"| Metric   | Value    |\n"
    f"|----------|----------|\n"
    f"| HR@1     | {test_hr1*100:.2f}%  |\n"
    f"| HR@5     | {test_hr5*100:.2f}%  |\n"
    f"| HR@10    | {test_hr10*100:.2f}%  |\n"
    f"| NDCG@10  | {test_ndcg10*100:.2f}%  |\n"
    f"| MRR      | {test_mrr*100:.2f}%  |\n"
    f"\nBest val HR@10: {best_val_hr10*100:.2f}% @ iter {best_iter}\n"
    f"Checkpoint: {CHECKPOINT_PATH}\n"
)

if state_path.exists():
    existing = state_path.read_text(encoding="utf-8")
    marker = f"## {NOTEBOOK_NAME}"
    if marker in existing:
        lines = existing.split("\n")
        new_lines = []
        skip = False
        for line in lines:
            if line.startswith(marker):
                skip = True
            elif skip and line.startswith("## ") and not line.startswith(marker):
                skip = False
            if not skip:
                new_lines.append(line)
        updated = "\n".join(new_lines) + result_block
    else:
        updated = existing + result_block
    state_path.write_text(updated, encoding="utf-8")
    print(f"[CELL 11-14] PROJECT_STATE.md updated: {state_path}")
else:
    state_path.write_text(result_block, encoding="utf-8")
    print(f"[CELL 11-14] PROJECT_STATE.md created: {state_path}")

cell_end("CELL 11-14", t0)


[CELL 11-14] Update PROJECT_STATE.md
[CELL 11-14] start=2026-04-10T10:21:51
[CELL 11-14] PROJECT_STATE.md updated: /home/user/jamalla/anonymous-users-mooc-session-meta/PROJECT_STATE.md
[CELL 11-14] elapsed=0.02s  done


In [16]:
# [CELL 11-15] Final comparison table — all 4 MAML variants
#
# Attempt to load NB07/NB08/NB10 results from their report JSONs.
# If not found, placeholders are shown. Fill after running those notebooks.


def load_best_metrics(notebook_name: str) -> Optional[Dict]:
    """Load test metrics from the most recent report.json for a notebook."""
    reports_base = PATHS["REPORTS"] / notebook_name
    if not reports_base.exists():
        return None
    run_dirs = sorted(reports_base.iterdir(), reverse=True)
    for run_dir in run_dirs:
        report_file = run_dir / "report.json"
        if report_file.exists():
            try:
                rpt = read_json(report_file)
                if rpt.get("metrics") and rpt["metrics"].get("hr10") is not None:
                    return rpt["metrics"]
            except Exception:
                continue
    return None


def fmt_pct(v: Optional[float]) -> str:
    return f"{v*100:.2f}%" if v is not None else "—"


# Try to load prior notebook results
nb07_metrics = load_best_metrics("07_standard_maml_mars")
nb08_metrics = load_best_metrics("08_warmstart_maml_mars")
nb10_metrics = load_best_metrics("10_srs_adaptive_maml_mars")

def get_m(d, key):
    return d[key] if d else None

# Load best_iter for each (from results JSON if available)
def load_best_iter(notebook_name: str) -> Optional[int]:
    reports_base = PATHS["REPORTS"] / notebook_name
    if not reports_base.exists():
        return None
    run_dirs = sorted(reports_base.iterdir(), reverse=True)
    for run_dir in run_dirs:
        results_file = run_dir / f"results_{notebook_name}.json"
        if results_file.exists():
            try:
                res = read_json(results_file)
                return res.get("val_best", {}).get("best_iter")
            except Exception:
                continue
    return None


nb07_iter = load_best_iter("07_standard_maml_mars")
nb08_iter = load_best_iter("08_warmstart_maml_mars")
nb10_iter = load_best_iter("10_srs_adaptive_maml_mars")


def fmt_iter(v):
    return str(v) if v is not None else "—"


print("=" * 68)
print(f"FINAL RESULTS — {DATASET.upper()}")
print("=" * 68)
print(f"Protocol: K={K_SUPPORT} support, Q={Q_QUERY} query | Test: {len(episodes_test)} episodes")
print()
print(f"{'Model':<36} {'HR@10':>7} {'NDCG@10':>9} {'MRR':>7} {'Best Iter':>10}")
print("-" * 68)
print(
    f"{'Standard MAML        (NB07)':<36} "
    f"{fmt_pct(get_m(nb07_metrics,'hr10')):>7} "
    f"{fmt_pct(get_m(nb07_metrics,'ndcg10')):>9} "
    f"{fmt_pct(get_m(nb07_metrics,'mrr')):>7} "
    f"{fmt_iter(nb07_iter):>10}"
)
print(
    f"{'Warm-Start MAML      (NB08)':<36} "
    f"{fmt_pct(get_m(nb08_metrics,'hr10')):>7} "
    f"{fmt_pct(get_m(nb08_metrics,'ndcg10')):>9} "
    f"{fmt_pct(get_m(nb08_metrics,'mrr')):>7} "
    f"{fmt_iter(nb08_iter):>10}"
)
print(
    f"{'SRS-Adaptive MAML    (NB10)':<36} "
    f"{fmt_pct(get_m(nb10_metrics,'hr10')):>7} "
    f"{fmt_pct(get_m(nb10_metrics,'ndcg10')):>9} "
    f"{fmt_pct(get_m(nb10_metrics,'mrr')):>7} "
    f"{fmt_iter(nb10_iter):>10}"
)
print(
    f"{'Warm-Start+SRS-Adapt (NB11)':<36} "
    f"{test_hr10*100:>6.2f}% "
    f"{test_ndcg10*100:>8.2f}% "
    f"{test_mrr*100:>6.2f}% "
    f"{best_iter:>10}   <- MAIN"
)
print("=" * 68)
print("Note: Fill NB07/NB08/NB10 values from their respective report.json after running.")
print("      Dashes (—) indicate that notebook has not been run yet.")

FINAL RESULTS — MARS
Protocol: K=5 support, Q=10 query | Test: 17 episodes

Model                                  HR@10   NDCG@10     MRR  Best Iter
--------------------------------------------------------------------
Standard MAML        (NB07)           17.65%    15.55%  15.36%        100
Warm-Start MAML      (NB08)           27.06%    23.01%  22.29%        100
SRS-Adaptive MAML    (NB10)                —         —       —          —
Warm-Start+SRS-Adapt (NB11)           27.06%    21.83%  20.72%        100   <- MAIN
Note: Fill NB07/NB08/NB10 values from their respective report.json after running.
      Dashes (—) indicate that notebook has not been run yet.


## Notebook 11 Complete — Warm-Start + SRS-Adaptive MAML (MARS)

**MAIN CONTRIBUTION (C3)** — combines warm-start initialisation with SRS-Adaptive inner loop.

**Results (17 test episodes, 170 query predictions):**

```
========== FINAL RESULTS — MARS ==========
Protocol: K=5 support, Q=10 query | Test: 17 episodes

Model                            HR@10    NDCG@10  MRR     Best Iter
------------------------------------------------------------------
Standard MAML        (NB07)     17.65%   15.55%   15.36%  100
Warm-Start MAML      (NB08)     27.06%   23.01%   22.29%  100
SRS-Adaptive MAML    (NB10)     18.24%   16.17%   15.90%  100
Warm-Start+SRS-Adapt (NB11)     27.06%   21.83%   20.72%  100   <- MAIN
==========================================
```

**Config (C3 — all three differences from NB07):**
```
outer_lr     = 0.0001   <- KEY DIFF #2: 10x lower than NB07 (0.001)
warm_start   = True     <- KEY DIFF #1: loaded pretrained gru_global_mars.pth
srs_adaptive = True     <- KEY DIFF #3: task-specific alpha_i and K_i
alpha_base=0.01  tau=0.5  K_min=3  K_max=7
batch_size=32  n_iterations=3000  seed=20260107
```

